# Annexe B — Cahier de code, Chapitre 12
## Seuillage et segmentation

Ce notebook accompagne le chapitre 12 de *Fondamentaux de la Vision par
Ordinateur*. Pour **chaque sous-chapitre**, une cellule de code Python montre la
**syntaxe et l'usage** de la notion — au plus simple, sans fonctions ni gestion
d'erreurs. Le but n'est pas de produire du code de production, mais de relier la
formule du livre à son équivalent Python.

> **Pré-requis** : `pip install numpy scipy scikit-image scikit-learn opencv-python matplotlib`

Exécutez les cellules dans l'ordre : la première prépare les données d'entrée.

## Préparation des données

In [ ]:
# Image d'exemple en niveaux de gris
import numpy as np
from skimage import data, color
import matplotlib.pyplot as plt

img = color.rgb2gray(data.astronaut())   # float 0..1, H×W
plt.imshow(img, cmap="gray"); plt.title("image (gris)"); plt.show()

## 12.1 — Seuillage d'Otsu

In [ ]:
# choisit automatiquement le seuil qui sépare le mieux deux modes
from skimage.filters import threshold_otsu
seuil = threshold_otsu(img)
binaire = img > seuil

## 12.2 — Seuillage adaptatif

In [ ]:
# un seuil différent par voisinage (utile si l'éclairage varie)
from skimage.filters import threshold_local
seuil_local = threshold_local(img, block_size=35, offset=0.02)
binaire = img > seuil_local

## 12.3 — Watershed

In [ ]:
# inonder le relief depuis des marqueurs (sépare les objets collés)
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from scipy.ndimage import distance_transform_edt, label
mask = img > threshold_otsu(img) if 'threshold_otsu' in dir() else img > 0.5
dt = distance_transform_edt(mask)
marqueurs = label(np.zeros_like(mask))[0]
coords = peak_local_max(dt, labels=mask, min_distance=20)
marqueurs[tuple(coords.T)] = np.arange(1, len(coords) + 1)
labels = watershed(-dt, marqueurs, mask=mask)

## 12.4 — K-means

In [ ]:
# regrouper les pixels par valeur en k classes
from sklearn.cluster import KMeans
X = img.reshape(-1, 1)
km = KMeans(n_clusters=3, n_init=10).fit(X)
seg = km.labels_.reshape(img.shape)

## 12.5 — Mean-shift

In [ ]:
# remonte vers les pics de densité, sans fixer le nombre de classes
import cv2
from skimage import data
col = data.astronaut()
seg = cv2.pyrMeanShiftFiltering(col, sp=20, sr=30)

## 12.6 — Contours actifs (snakes)

In [ ]:
# une courbe élastique attirée par les bords
from skimage.segmentation import active_contour
from skimage.filters import gaussian
s = np.linspace(0, 2 * np.pi, 200)
init = np.column_stack([256 + 180 * np.sin(s), 256 + 180 * np.cos(s)])
snake = active_contour(gaussian(img, 3), init)

## 12.7 — Coupe de graphe (graph cut)

In [ ]:
# décision globale via marqueurs (random walker, accessible dans skimage)
from skimage.segmentation import random_walker
marqueurs = np.zeros(img.shape, dtype=int)
marqueurs[img < 0.3] = 1        # graines « fond »
marqueurs[img > 0.7] = 2        # graines « objet »
labels = random_walker(img, marqueurs)